In [16]:
import time
from functools import wraps

import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
plt.show

from classes import InvalidViolationDataError
from data import build_drivers_and_records

# 1. MEASURING THE EXECUTION TIME
def timed(func):
    """ Measures and prints the execution time of a function. """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter() # this is used for measuring elapsed time: .perf_counter is Python's standard lib func
        result = func(*args, *kwargs)
        elapsed_time = time.perf_counter() - start_time
        print(f"[Execution Time: {func.__name__:<28} {elapsed_time:.6f} s]")
        return result
    return wrapper

# function above basically executes start time -> run function -> calculate elapsed time

# 2. DATA VALIDATION

REQUIRED_COLUMNS = [
    "Driver ID", "Violation Type", "Fine Amount", "Speed Over Limit", 
    "Number of Violation", "Payment Status", "Violation Month",
]

@timed
def validate_dataframe(df):
    """
        Validates the created DataFrame based on the assessment's data requirements.
        Raises an error if invalid data is found (InvalidViolationDataError).
    """
    missing_columns = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_columns:
        raise InvalidViolationDataError(f"Missing required columns: {missing_columns}")

    if len(df) < 50:
        raise InvalidViolationDataError(f"Dataset has only {len(df)} records; the assessment brief requires at least 50 records.")

    if df["Fine Amount"].lt(0).any():
        raise InvalidViolationDataError("One or more fine amounts are found negative.")

    if df["Speed Over Limit"].lt(0).any():
        raise InvalidViolationDataError("One or more speed-over-limit values are found negative.")

    invalid_months = df.loc[~df["Violation Month"].between(1, 12), "Violation Month"]
    if not invalid_months.empty:
        raise InvalidViolationDataError(f"Invalid violation month(s): {invalid_months.tolist()}")

    invalid_status = set(df["Payment Status"].unique()) - {"Paid", "Unpaid"}
    if invalid_status:
        raise InvalidViolationDataError(f"Invalid payment status value(s): {invalid_status}")

    inconsistent_records = df[(df["Violation Type"] != "Speeding") & (df["Speed Over Limit"] != 0)]
    if not inconsistent_records.empty:
        raise InvalidViolationDataError(f"{len(inconsistent_records)} non-speeding record(s) have a non-zero speed over limit")

    print(f"[VALIDATION] Data validation successful: {len(df)} records, {len(df.columns)} columns.")
    return True

# BUILDING THE DATAFRAME
@timed
def build_dataframe():
    """ Builds the records and converts them into a Pandas DataFrame. """
    try:
        drivers, records = build_drivers_and_records()
    except Exception as error:
        raise InvalidViolationDataError(f"Unable to build records: {error}")

    df = pd.DataFrame(records)
    # Keep only the required columns in the required order based on assessment brief
    df = df[[c for c in REQUIRED_COLUMNS if c in df.columns]]
    return df, drivers

# DATA ANALYSIS
@timed
def summary_statistics(df):
    """Summary of the statistics in a descriptive way."""
    speeding_records = df[df["Speed Over Limit"] > 0]
    return {
        "Total number of records": len(df),
        "Unique drivers": df["Driver ID"].nunique(),
        "Total fines issued": round(df["Fine Amount"].sum(), 2),
        "Fines collected (Paid)": round(df.loc[df["Payment Status"] == "Paid", "Fine Amount"].sum(), 2),
        "Fines outstanding (Unpaid)": round(df.loc[df["Payment Status"] == "Unpaid", "Fine Amount"].sum(), 2),
        "Average fine": round(df["Fine Amount"].mean(), 2),
        "Most common violation": df["Violation Type"].mode()[0],
        "Average speed over limit (speeding only)": round(speeding_records["Speeding Over Limit"].mean(), 2),
        "Unpaid rate (%)": round((df["Payment Status"] == "Unpaid").mean() * 100, 1),
    }

@timed
def find_repeat_offenders(df, threshold=3):
    """Finds drivers with three or more violations."""
    violation_counts = df.groupby("Driver ID").size().sort_values(ascending=False)
    return violation_counts[violation_counts >= threshold]